In [26]:
import pandas as pd
import numpy as np
import glob
import os

# ---- PATHS ----
BASE      = r'D:\FYP\Eco-Twin'
RAW_PATH  = os.path.join(BASE, 'Raw Sets')
V2_DIR    = os.path.join(BASE, 'v2_clean')
os.makedirs(V2_DIR, exist_ok=True)

STATIONS_EXCEL_IN  = os.path.join(BASE, 'Delhi_list.xlsx')          # original 44-station list
STATIONS_EXCEL_OUT = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')     # cleaned 36-station list
DATASET_OUT        = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIC_ADJ_OUT     = os.path.join(V2_DIR, 'static_adj_matrix.npy')

# ---- 8 stations to drop (from raw missingness diagnostic) ----
DROP_STATIONS = [
    'Talkatora Garden', 'IGNOU_Maidan Garhi', 'Cantonment Area',
    'NSUT Jaffarpur', 'Commonwealth Sports Complex', 'JNU',
    'IIT Delhi', 'New Moti Bagh'
]

THRESHOLD_KM = 12.0

REQUIRED_COLS = ['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'ws', 'wd', 'at']

print(f"Raw files path : {RAW_PATH}")
print(f"Output dir     : {V2_DIR}")
print(f"Dropping {len(DROP_STATIONS)} stations:")
for s in DROP_STATIONS:
    print(f"  - {s}")

Raw files path : D:\FYP\Eco-Twin\Raw Sets
Output dir     : D:\FYP\Eco-Twin\v2_clean
Dropping 8 stations:
  - Talkatora Garden
  - IGNOU_Maidan Garhi
  - Cantonment Area
  - NSUT Jaffarpur
  - Commonwealth Sports Complex
  - JNU
  - IIT Delhi
  - New Moti Bagh


In [27]:
df_stations = pd.read_excel(STATIONS_EXCEL_IN)
df_stations.columns = df_stations.columns.str.strip().str.lower()
df_stations['station'] = df_stations['station'].str.strip()

# validate coordinate columns
for col in ['station', 'latitude', 'longitude']:
    assert col in df_stations.columns, f"Missing '{col}' in {STATIONS_EXCEL_IN}"

before = len(df_stations)
df_stations = df_stations[~df_stations['station'].isin(DROP_STATIONS)].reset_index(drop=True)
after = len(df_stations)

# save cleaned station list
df_stations.to_excel(STATIONS_EXCEL_OUT, index=False)

station_names = df_stations['station'].tolist()
N = len(station_names)

print(f"Stations before : {before}")
print(f"Stations after  : {after}")
print(f"Saved cleaned list -> {STATIONS_EXCEL_OUT}")
print(f"\nKeeping {N} stations.")

Stations before : 45
Stations after  : 37
Saved cleaned list -> D:\FYP\Eco-Twin\v2_clean\Delhi_list_v2.xlsx

Keeping 37 stations.


In [28]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

lats = df_stations['latitude'].values
lons = df_stations['longitude'].values

dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])

# binary adjacency, no self-loops
static_adj = ((dist_matrix > 0) & (dist_matrix <= THRESHOLD_KM)).astype(int)
np.save(STATIC_ADJ_OUT, static_adj)

# neighbour report
neighbor_map = {
    name: [station_names[j] for j in np.where(static_adj[i] == 1)[0]]
    for i, name in enumerate(station_names)
}
counts = [len(v) for v in neighbor_map.values()]

print(f"Static adjacency saved -> {STATIC_ADJ_OUT}")
print(f"Threshold        : {THRESHOLD_KM} km")
print(f"Avg neighbours   : {np.mean(counts):.1f}")
print(f"Min / Max        : {np.min(counts)} / {np.max(counts)}")
isolated = [station_names[i] for i, c in enumerate(counts) if c == 0]
if isolated:
    print(f"\n⚠ Isolated stations (no neighbours): {isolated}")
    print("  Consider raising THRESHOLD_KM.")
else:
    print(f"\n✓ Every station has at least one neighbour.")

Static adjacency saved -> D:\FYP\Eco-Twin\v2_clean\static_adj_matrix.npy
Threshold        : 12.0 km
Avg neighbours   : 13.7
Min / Max        : 1 / 24

✓ Every station has at least one neighbour.


In [29]:
COL_MAP = {
    'PM2.5': 'pm2.5', 'PM10': 'pm10', 'NO2': 'no2', 'SO2': 'so2',
    'CO': 'co', 'Ozone': 'o3', 'O3': 'o3', 'RH': 'rh',
    'WS': 'ws', 'WD': 'wd', 'AT': 'at', 'Temp': 'at',
    'Ambient Temperature': 'at', 'Temperature': 'at',
}
NULL_STRINGS = ['None', 'N/A', '-', '--', ' ', 'null', 'NA', 'nan', 'NaN']

PHYSICAL_LIMITS = {
    'pm2.5': (0, 1000), 'pm10': (0, 2000), 'no2': (0, 500),
    'so2': (0, 500), 'co': (0, 10000), 'o3': (0, 500),
    'rh': (0, 100), 'ws': (0, 50), 'wd': (0, 360), 'at': (-20, 55),
}

all_files = glob.glob(os.path.join(RAW_PATH, '*.xls*'))
master_list = []

for filepath in sorted(all_files):
    station = os.path.basename(filepath).rsplit('.', 1)[0].strip()
    if station in DROP_STATIONS:
        continue   # skip dropped stations entirely
    if station not in station_names:
        print(f"  [skip] {station} — not in cleaned station list")
        continue
    try:
        engine = 'xlrd' if filepath.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(filepath, engine=engine)
        df.replace(NULL_STRINGS, np.nan, inplace=True)
        df.rename(columns=COL_MAP, inplace=True)

        for ts in ('From Date', 'Date', 'Timestamp', 'DateTime'):
            if ts in df.columns:
                df.rename(columns={ts: 'Timestamp'}, inplace=True)
                break
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')
        df['Station'] = station

        for col in REQUIRED_COLS:
            if col not in df.columns:
                df[col] = np.nan
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # physical clipping
        for col, (lo, hi) in PHYSICAL_LIMITS.items():
            df[col] = df[col].where(df[col].between(lo, hi))

        df = df[['Timestamp', 'Station'] + REQUIRED_COLS].dropna(subset=['Timestamp'])
        df = df.groupby(['Timestamp', 'Station'], as_index=False).mean(numeric_only=True)

        master_list.append(df)
        print(f"  [ok] {station} ({len(df)} rows)")
    except Exception as e:
        print(f"  [FAIL] {station}: {e}")

master_df = pd.concat(master_list, ignore_index=True)
master_df.sort_values(['Station', 'Timestamp'], inplace=True)
print(f"\nCombined rows: {len(master_df):,}")
print(f"Stations loaded: {master_df['Station'].nunique()} (expect {N})")

  [ok] Alipur (19921 rows)
  [ok] Anand Vihar (19921 rows)
  [ok] Ashok Vihar (19921 rows)
  [ok] Aya Nagar (19921 rows)
  [ok] Bawana (19921 rows)
  [ok] Burari Crossing (19921 rows)
  [ok] CRRI Mathura Road (19921 rows)
  [ok] Chandni Chowk (19921 rows)
  [ok] DTU (19921 rows)
  [ok] Dr. Karni Singh Shooting Range (19921 rows)
  [ok] Dwarka-Sector 8 (19921 rows)
  [ok] IGI Airport (T3) (19921 rows)
  [ok] ITO (19921 rows)
  [ok] Jahangirpuri (19921 rows)
  [ok] Jawaharlal Nehru Stadium (19921 rows)
  [ok] Lodhi Road (19921 rows)
  [ok] Major Dhyan Chand National Stadium (19921 rows)
  [ok] Mandir Marg (19921 rows)
  [ok] Mundka (19921 rows)
  [ok] NSIT Dwarka (19921 rows)
  [ok] Najafgarh (19921 rows)
  [ok] Narela (19921 rows)
  [ok] Nehru Nagar (19921 rows)
  [ok] North Campus (19921 rows)
  [ok] Okhla Phase-2 (19921 rows)
  [ok] Patparganj (19921 rows)
  [ok] Punjabi Bagh (19921 rows)
  [ok] Pusa DPCC (19921 rows)
  [ok] Pusa IITM (19921 rows)
  [ok] R K Puram (19921 rows)
  [ok] 

In [31]:
t_min = master_df['Timestamp'].min()
t_max = master_df['Timestamp'].max()

station_frames = []
for station, grp in master_df.groupby('Station'):
    g = grp.set_index('Timestamp').sort_index()[REQUIRED_COLS]
    hourly = g.resample('1h').mean()
    hourly['Station'] = station
    station_frames.append(hourly.reset_index())

master_df = pd.concat(station_frames, ignore_index=True)
master_df.sort_values(['Timestamp', 'Station'], inplace=True)

print(f"After resampling: {len(master_df):,} rows")
print(f"Unique hours    : {master_df['Timestamp'].nunique():,}")
print(f"Date range      : {master_df['Timestamp'].min()} -> {master_df['Timestamp'].max()}")

After resampling: 737,077 rows
Unique hours    : 19,921
Date range      : 2024-01-01 00:00:00 -> 2026-04-10 00:00:00


In [32]:
def vectorised_spatial_fill(df, feature, neighbor_map):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=feature, aggfunc='first')
    filled = wide.copy()
    for station, neighbours in neighbor_map.items():
        if station not in wide.columns:
            continue
        valid = [n for n in neighbours if n in wide.columns]
        if not valid:
            continue
        filled[station] = wide[station].fillna(wide[valid].mean(axis=1))
    try:
        long = filled.stack(future_stack=True).reset_index()
    except TypeError:
        long = filled.stack().reset_index()
    long.columns = ['Timestamp', 'Station', feature]
    merged = df[['Timestamp', 'Station']].merge(long, on=['Timestamp', 'Station'], how='left')
    return merged[feature].values

for feat in REQUIRED_COLS:
    miss_before = master_df[feat].isna().sum()
    if miss_before == 0:
        continue
    print(f"Imputing {feat:<6} (missing {miss_before:,})")
    # Tier 1 — spatial neighbours
    master_df[feat] = vectorised_spatial_fill(master_df, feat, neighbor_map)
    # Tier 2 — timestamp-wise city mean
    master_df[feat] = master_df[feat].fillna(
        master_df.groupby('Timestamp')[feat].transform('mean'))
    # Tier 3 — temporal interpolation per station
    master_df[feat] = master_df.groupby('Station')[feat].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both'))
    # Tier 4 — ffill/bfill for any residual
    master_df[feat] = master_df.groupby('Station')[feat].transform(lambda x: x.ffill().bfill())
    print(f"   -> remaining NaN: {master_df[feat].isna().sum():,}")

print(f"\nTotal residual NaN: {master_df[REQUIRED_COLS].isna().sum().sum()}")

Imputing pm2.5  (missing 39,564)
   -> remaining NaN: 0
Imputing pm10   (missing 33,979)
   -> remaining NaN: 0
Imputing no2    (missing 27,312)
   -> remaining NaN: 0
Imputing so2    (missing 179,644)
   -> remaining NaN: 0
Imputing co     (missing 32,091)
   -> remaining NaN: 0
Imputing o3     (missing 70,327)
   -> remaining NaN: 0
Imputing rh     (missing 179,146)
   -> remaining NaN: 0
Imputing ws     (missing 210,683)
   -> remaining NaN: 0
Imputing wd     (missing 194,272)
   -> remaining NaN: 0
Imputing at     (missing 214,317)
   -> remaining NaN: 0

Total residual NaN: 0


In [33]:
master_df['wd'] = master_df['wd'] % 360
wd_rad = np.deg2rad(master_df['wd'])
master_df['u_wind'] = -master_df['ws'] * np.sin(wd_rad)
master_df['v_wind'] = -master_df['ws'] * np.cos(wd_rad)

print("Wind vectorisation done.")
print(f"  u_wind range: [{master_df['u_wind'].min():.2f}, {master_df['u_wind'].max():.2f}]")
print(f"  v_wind range: [{master_df['v_wind'].min():.2f}, {master_df['v_wind'].max():.2f}]")
print(f"  u_wind std  : {master_df['u_wind'].std():.3f}")
print(f"  v_wind std  : {master_df['v_wind'].std():.3f}")

Wind vectorisation done.
  u_wind range: [-24.31, 33.43]
  v_wind range: [-35.14, 35.30]
  u_wind std  : 0.924
  v_wind std  : 0.952


In [36]:
output_cols = ['Timestamp', 'Station'] + REQUIRED_COLS + ['u_wind', 'v_wind']
master_df[output_cols].to_csv(DATASET_OUT, index=False)

print(f"  Dataset saved  : {DATASET_OUT}")
print(f"  Rows           : {len(master_df):,}")
print(f"  Stations       : {master_df['Station'].nunique()}")
print(f"  Hours          : {master_df['Timestamp'].nunique():,}")
print(f"  Columns        : {output_cols}")
print()
print("  Also saved:")
print(f"    {STATIONS_EXCEL_OUT}")
print(f"    {STATIC_ADJ_OUT}")
print()

  Dataset saved  : D:\FYP\Eco-Twin\v2_clean\EcoTwin_Final_Dataset.csv
  Rows           : 737,077
  Stations       : 37
  Hours          : 19,921
  Columns        : ['Timestamp', 'Station', 'pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'ws', 'wd', 'at', 'u_wind', 'v_wind']

  Also saved:
    D:\FYP\Eco-Twin\v2_clean\Delhi_list_v2.xlsx
    D:\FYP\Eco-Twin\v2_clean\static_adj_matrix.npy

